# Task 4 — Integrated Analysis of State Real GDP & Labour Force (Malaysia, 2023)
**Course:** STQD6014 Data Science — Project Report 2  
**Author:** Mhamad Shhab Aldeen Hasan · P166175  
**Datasets:**
1. `gdp_state_real_supply.csv` — Annual Real GDP by State & Economic Sector (DOSM)
2. `lfs_state_sex.csv` — Annual Labour Force Statistics by State & Sex (DOSM)

**Research questions:**
- How does the sector composition of state GDP relate to labour force participation rates?
- Which states have the highest unemployment, and is that linked to their economic structure?
- What is the gender gap in labour force participation, and which economic structures narrow it?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Load raw datasets
gdp    = pd.read_csv('data/gdp_state_real_supply.csv')
labour = pd.read_csv('data/lfs_state_sex.csv')

print("GDP    shape:", gdp.shape)
print("Labour shape:", labour.shape)
gdp.head()

## Section 1 — Data Preprocessing & Integration

In [ ]:
# ── 1a. Filter to 2023 ───────────────────────────────────────────────────────
gdp_2023    = gdp[(gdp['series'] == 'abs') &
                  (gdp['date'].astype(str).str.startswith('2023'))].copy()
labour_2023 = labour[labour['date'].astype(str).str.startswith('2023')].copy()

print(f"GDP 2023 records    : {len(gdp_2023)}")
print(f"Labour 2023 records : {len(labour_2023)}")

# ── 1b. Harmonise state names ─────────────────────────────────────────────────
# GDP dataset uses 'Supra' for what labour dataset calls 'W.P. Putrajaya'
# (The 'Supra' row captures import duties not attributable to any single state;
#  we map it to W.P. Putrajaya for the join but flag it as anomalous.)
gdp_2023['state'] = gdp_2023['state'].replace({'Supra': 'W.P. Putrajaya'})

print("\nUnique GDP states  :", sorted(gdp_2023['state'].unique()))
print("Unique LFS states  :", sorted(labour_2023['state'].unique()))

In [ ]:
# ── 1c. Pivot GDP to wide format (one row per state, columns = sectors) ──────
gdp_wide = (
    gdp_2023
    .pivot_table(values='value', index='state', columns='sector', aggfunc='first')
    .reset_index()
)

# ── 1d. Pivot Labour to wide format ───────────────────────────────────────────
metrics = ['lf', 'lf_employed', 'lf_unemployed', 'lf_outside',
           'p_rate', 'u_rate', 'ep_ratio']
labour_wide = labour_2023.pivot_table(
    index='state', columns='sex', values=metrics, aggfunc='first'
)
labour_wide.columns = [f"{m}_{s}" for m, s in labour_wide.columns]
labour_wide = labour_wide.reset_index()

print("GDP wide  shape:", gdp_wide.shape)
print("LFS wide  shape:", labour_wide.shape)
gdp_wide.head(3)

In [ ]:
# ── 1e. Inner join on state → 16 rows ────────────────────────────────────────
df = pd.merge(gdp_wide, labour_wide, on='state', how='inner')
print(f"Merged rows: {len(df)} states")

# ── 1f. Sector proportions (% of total GDP) ───────────────────────────────────
sector_map = {
    'p0': 'Total GDP',
    'p1': 'Agriculture',
    'p2': 'Mining & Quarrying',
    'p3': 'Manufacturing',
    'p4': 'Construction',
    'p5': 'Services',
    'p6': 'Taxes less subsidies'
}
for s in ['p1', 'p2', 'p3', 'p4', 'p5', 'p6']:
    df[f'{s}_prop'] = df[s] / df['p0'] * 100

# ── 1g. GDP per employed person (RM) ──────────────────────────────────────────
# GDP in million RM; employment in thousands of persons → convert to RM per person
df['gdp_per_employed_rm'] = (df['p0'] * 1e6) / (df['lf_employed_both'] * 1e3)

# Flag the Putrajaya record (mapped from 'Supra') — sector split is anomalous
df['putrajaya_flag'] = df['state'] == 'W.P. Putrajaya'

print("Columns:", df.columns.tolist())
df[['state', 'p0', 'lf_employed_both', 'u_rate_both', 'gdp_per_employed_rm']].head(5)

## Section 2 — Descriptive Statistics (2023)

In [ ]:
summary_cols = ['p0', 'p5_prop', 'p3_prop', 'p2_prop',
                'lf_both', 'lf_employed_both', 'u_rate_both',
                'p_rate_both', 'gdp_per_employed_rm']
df[summary_cols].describe().round(2)

In [ ]:
print("Top 5 states by Total Real GDP (million RM):")
print(df[['state', 'p0']].sort_values('p0', ascending=False).head().to_string(index=False))
print()
print("Top 5 unemployment rates (%):")
print(df[['state', 'u_rate_both']].sort_values('u_rate_both', ascending=False).head().to_string(index=False))
print()
print("Top 5 female labour force participation rates (%):")
print(df[['state', 'p_rate_female']].dropna().sort_values('p_rate_female', ascending=False).head().to_string(index=False))

## Section 3 — Visualisations
### Figure 1 — GDP Sector Composition by State

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

plot_df = df.sort_values('p0', ascending=False).set_index('state')
shares  = plot_df[[f'{s}_prop' for s in ['p1','p2','p3','p4','p5','p6']]]
shares.columns = [sector_map[c.replace('_prop', '')] for c in shares.columns]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
shares.plot(kind='bar', stacked=True, figsize=(14, 6), color=colors, ax=ax)

ax.set_ylabel("Share of 2023 real GDP (%)")
ax.set_title("Malaysia 2023 Real GDP Composition by State (sector shares)")
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figures/fig1_gdp_sector_shares.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/fig1_gdp_sector_shares.png")

### Figure 2 — Labour Force Participation Rate by Sex

In [ ]:
# Exclude Negeri Sembilan: male/female breakdown missing in 2023
plot_df2 = (df[['state', 'p_rate_male', 'p_rate_female']]
            .dropna()
            .sort_values('p_rate_female', ascending=True))

x     = np.arange(len(plot_df2))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width/2, plot_df2['p_rate_male'],   width, label='Male LFPR',   color='#1f77b4')
ax.bar(x + width/2, plot_df2['p_rate_female'], width, label='Female LFPR', color='#ff7f0e')
ax.set_xticks(x)
ax.set_xticklabels(plot_df2['state'], rotation=45, ha='right')
ax.set_ylabel("Labour Force Participation Rate (%)")
ax.set_title("Labour Force Participation Rate by State and Sex (2023)\n"
             "(Excludes Negeri Sembilan due to missing sex breakdown)")
ax.legend()
plt.tight_layout()
plt.savefig('figures/fig2_lfpr_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/fig2_lfpr_gender.png")

### Figure 3 — Services Share vs Female LFPR

In [ ]:
df_no_pj = df[~df['putrajaya_flag']].copy()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df_no_pj['p5_prop'], df_no_pj['p_rate_female'], s=60)

for _, r in df_no_pj.iterrows():
    if r['state'] in ['W.P. Kuala Lumpur', 'Selangor', 'Kelantan', 'Sabah']:
        ax.annotate(r['state'].replace('W.P. ', 'WP '),
                    (r['p5_prop'], r['p_rate_female']),
                    fontsize=8, xytext=(5, 5), textcoords='offset points')

r = df_no_pj[['p5_prop', 'p_rate_female']].dropna().corr().iloc[0, 1]
ax.set_xlabel("Services prop of GDP (%)")
ax.set_ylabel("Female LFPR (%)")
ax.set_title("Services sector share vs female labour force participation (2023)\n"
             "(Excludes W.P. Putrajaya due to GDP-sector anomaly)")
ax.annotate(f"r = {r:.2f}", xy=(0.05, 0.92), xycoords='axes fraction', fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig3_services_vs_female_lfpr.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Pearson r (services ~ female LFPR): {r:.3f}")
print("Saved: figures/fig3_services_vs_female_lfpr.png")

### Figure 4 — GDP per Employed Person vs Unemployment Rate

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df_no_pj['gdp_per_employed_rm'], df_no_pj['u_rate_both'], s=60)

for _, r in df_no_pj.iterrows():
    if r['state'] in ['Sabah', 'W.P. Labuan', 'W.P. Kuala Lumpur', 'Selangor', 'Kelantan']:
        ax.annotate(r['state'].replace('W.P. ', 'WP '),
                    (r['gdp_per_employed_rm'], r['u_rate_both']),
                    fontsize=8, xytext=(5, 5), textcoords='offset points')

ax.set_xlabel("GDP per employed person (RM, 2015 prices)")
ax.set_ylabel("Unemployment rate (%)")
ax.set_title("GDP per employed person vs unemployment rate (2023)\n"
             "(Excludes W.P. Putrajaya due to GDP-sector anomaly)")
plt.tight_layout()
plt.savefig('figures/fig4_productivity_vs_unemployment.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: figures/fig4_productivity_vs_unemployment.png")

## Section 4 — Correlation Analysis

In [ ]:
corr_pairs = [
    ('p5_prop',   'p_rate_female',   'Services share',   'Female LFPR'),
    ('p2_prop',   'u_rate_both',     'Mining share',     'Unemployment rate'),
    ('p3_prop',   'u_rate_both',     'Manuf. share',     'Unemployment rate'),
    ('p_rate_both','u_rate_both',    'Overall LFPR',     'Unemployment rate'),
]

print(f"{'Variable A':<20} {'Variable B':<20} {'r':>6}")
print("-" * 50)
for va, vb, la, lb in corr_pairs:
    sub = df_no_pj[[va, vb]].dropna()
    r   = sub.corr().iloc[0, 1]
    print(f"{la:<20} {lb:<20} {r:+.3f}")

## Section 5 — Save Merged Dataset

In [ ]:
df.to_csv('outputs/merged_gdp_lfs_2023.csv', index=False)
print(f"Saved: outputs/merged_gdp_lfs_2023.csv  ({len(df)} rows x {len(df.columns)} cols)")
print()
print("Columns in merged dataset:")
for c in df.columns:
    print(f"  {c}")

## Key Findings

**Economic structure:**
- Selangor leads total real GDP (~RM406,071 M), followed by W.P. Kuala Lumpur (~RM249,302 M) and Johor (~RM148,205 M)
- Services dominate in urban states (KL >90%); Manufacturing is highest in Pulau Pinang (~46%)
- Agriculture is highest in Pahang (~22%) and Kelantan (~20%)

**Labour market:**
- Unemployment highest in Sabah (7.5%) and W.P. Labuan (6.8%)
- Labour force participation strongest in Selangor (77.4%) and W.P. Kuala Lumpur (74.2%)
- Gender gap large in all states; narrower in urban, service-oriented regions

**Sector–labour correlations:**
- Mining share ↔ Unemployment rate: r ≈ +0.50 (positive — mining states face higher unemployment)
- Manufacturing share ↔ Unemployment rate: r ≈ −0.52 (negative — manufacturing states have lower unemployment)
- Services share ↔ Female LFPR: r ≈ +0.18 (weak positive — urban services reduce but do not eliminate gender gap)

**Data challenges resolved:**
1. GDP 'Supra' mapped to 'W.P. Putrajaya' for join; flagged as anomalous in visualisations
2. Negeri Sembilan excluded from sex-specific plots (male/female rows missing in 2023)